# Chapter 7: Evaluation + decoding — measuring instead of eyeballing

Six chapters in, you have models — but every judgment about them has been a *vibe*: "looks more coherent", "still says Germany", "the loops are gone-ish". That's fine for intuition and useless for decisions. Before we add DPO (Ch.8), multilingual (Ch.9), and quantization (Ch.10), we need **numbers** — otherwise "did it help?" is unanswerable.

This chapter builds the three tools every LLM evaluation rests on:

1. **Held-out perplexity** — the intrinsic "how surprised is the model by real text" metric.
2. **Multiple-choice by log-likelihood** — *how base models are actually benchmarked* (ARC, HellaSwag, MMLU all work this way under the hood). No generation, no judge — just "which continuation does the model find most likely?"
3. **Decoding methods** — greedy vs temperature vs top-k vs **top-p (nucleus)** vs **min-p**, plus **repetition penalty** to finally kill the "AC AC AC" loops. Same model, very different text — decoding is a dial you've been leaving on default.

> **This is the foundation for the rest of the arc.** Ch.8 (DPO) and Ch.9 (multilingual) only mean something if you can *measure* the change. So: rigor first.

**Pattern unchanged:** concept → `TODO` → `# check`. The three measurement primitives — perplexity, choice log-likelihood, and the logit filters — are the TODOs.

## 0. Setup — load a model to evaluate

We evaluate the **Ch.4 modern base** (`modern.pt`). Starting here, the settled architecture and data utilities live in a shared **`llmscratch` package** at the repo root, so we just **import** them instead of re-pasting the classes into every chapter. (That copy-paste is also what let Ch.7's model drift from Ch.4's saved names and fail to load — one source of truth fixes that for good.)

> Optionally `pip install -e .` from the repo root to import `llmscratch` from anywhere; the cell below also works without it (it adds the repo root to `sys.path`). The chapters that *teach* a component (Ch.4 RoPE, Ch.6 MoE) still implement it inline — the package is for **reuse**, not for spoiling those lessons.

In [4]:
import math, os, time
from dataclasses import dataclass, replace

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
import tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
DATA_DIR = r"C:\Users\Dumii\Claude\Projects\LLM-from-scratch\chapters\chapter-4-modern-architecture\data"
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token
VOCAB_SIZE = enc.n_vocab
print("device:", device)

device: cuda


In [ ]:
# --- the settled Ch.4 model + data utils now live in the `llmscratch` package (no more copy-paste) ---
# This is the reusable fix for "every notebook re-pastes the model": one source of truth, so the
# class names always match the checkpoints. Robust import whether or not you ran `pip install -e .`:
try:
    from llmscratch.model import Config, LlamaGPT, generate
    from llmscratch.data import get_batch
except ModuleNotFoundError:
    import sys, pathlib
    _root = pathlib.Path.cwd()
    while _root != _root.parent and not (_root / "llmscratch").exists():
        _root = _root.parent                       # walk up to the repo root that holds llmscratch/
    sys.path.insert(0, str(_root))
    from llmscratch.model import Config, LlamaGPT, generate
    from llmscratch.data import get_batch

# importing Config also puts it in this notebook's namespace, so the pickled cfg in modern.pt unpickles
print("imported Config / LlamaGPT / generate / get_batch from llmscratch")

In [6]:
CKPT = os.path.join(DATA_DIR, "modern.pt")
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
model = LlamaGPT(ckpt["cfg"]).to(device)
model.load_state_dict(ckpt["model"]); model.eval()
print(f"loaded {CKPT}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params, block_size={model.block_size}")

RuntimeError: Error(s) in loading state_dict for LlamaGPT:
	Missing key(s) in state_dict: "blocks.0.n1.weight", "blocks.0.at.q_proj.weight", "blocks.0.at.k_proj.weight", "blocks.0.at.v_proj.weight", "blocks.0.at.o_proj.weight", "blocks.0.n2.weight", "blocks.0.m.gate.weight", "blocks.0.m.up.weight", "blocks.0.m.down.weight", "blocks.1.n1.weight", "blocks.1.at.q_proj.weight", "blocks.1.at.k_proj.weight", "blocks.1.at.v_proj.weight", "blocks.1.at.o_proj.weight", "blocks.1.n2.weight", "blocks.1.m.gate.weight", "blocks.1.m.up.weight", "blocks.1.m.down.weight", "blocks.2.n1.weight", "blocks.2.at.q_proj.weight", "blocks.2.at.k_proj.weight", "blocks.2.at.v_proj.weight", "blocks.2.at.o_proj.weight", "blocks.2.n2.weight", "blocks.2.m.gate.weight", "blocks.2.m.up.weight", "blocks.2.m.down.weight", "blocks.3.n1.weight", "blocks.3.at.q_proj.weight", "blocks.3.at.k_proj.weight", "blocks.3.at.v_proj.weight", "blocks.3.at.o_proj.weight", "blocks.3.n2.weight", "blocks.3.m.gate.weight", "blocks.3.m.up.w

## 1. Perplexity — the intrinsic metric

The training loss *is* the eval metric — cross-entropy on text the model didn't train on. **Perplexity** just makes it interpretable:

```
perplexity = exp(cross_entropy_loss)
```

Intuition: perplexity is the model's **effective branching factor** — "on average, how many equally-likely tokens is it choosing among?" A model that has learned *nothing* is uniformly unsure over the whole vocab → perplexity ≈ `vocab_size` (50257, loss ≈ `ln(V)` ≈ 10.8). A perfect model → perplexity 1 (it always knew the next token). Lower is better; halving perplexity is a big jump in quality.

**Held-out matters.** Perplexity on data the model *trained* on is optimistic (it partly memorized). A clean number uses a **reserved validation split** the model never saw. Our earlier chapters trained on the whole cache, so the split below isn't strictly held out — treat the *number* as a proxy and the *method* as the lesson. (For a real run: carve off `val` **before** training.)

In [ ]:
# reserve a tail slice as "val" (see caveat above) from any existing cache
_cands = ["mix_edu_100M.bin", "fineweb_train_500M.bin", "fineweb_train.bin", "fineweb_tiny.bin"]
_bin = next((os.path.join(DATA_DIR, c) for c in _cands if os.path.exists(os.path.join(DATA_DIR, c))), None)
assert _bin, "no token cache found -- run Ch.3a/4 first"
_all = np.memmap(_bin, dtype=np.uint16, mode="r")
val_data = _all[-2_000_000:]            # last 2M tokens as the eval slice
print(f"eval slice: {len(val_data):,} tokens from {_bin}")


@torch.no_grad()
def evaluate_perplexity(model, data, block, batch=16, iters=100):
    """Mean cross-entropy over `iters` random held-out batches, and its perplexity.
    Returns (mean_loss, perplexity). Steps:
      1. model.eval(); for `iters` steps: x, y = get_batch(data, block, batch, device)  # imported from llmscratch
         with autocast(bf16) + no grad, collect model(x, y)[1].item()
      2. mean_loss = average of the losses; perplexity = exp(mean_loss); return (mean_loss, perplexity)
    """
    # TODO:
    raise NotImplementedError

In [ ]:
# check -- perplexity == exp(loss), and a TRAINED model is far below the random-init baseline
_loss, _ppl = evaluate_perplexity(model, val_data, model.block_size, iters=50)
assert abs(_ppl - math.exp(_loss)) < 1e-3, "perplexity must equal exp(mean_loss)"

_rand = LlamaGPT(ckpt["cfg"]).to(device).eval()      # untrained: should be ~ln(V)/vocab_size
_rl, _rppl = evaluate_perplexity(_rand, val_data, model.block_size, iters=20)
print(f"trained:  loss {_loss:.3f}  perplexity {_ppl:7.1f}")
print(f"random:   loss {_rl:.3f}  perplexity {_rppl:7.1f}   (baseline ~ vocab {VOCAB_SIZE})")
assert _ppl < _rppl / 5, "a trained model must be vastly less perplexed than random init"
del _rand
print("ok: perplexity computed; training bought a big drop in branching factor")

## 2. Multiple-choice by log-likelihood — how base models are *really* benchmarked

You can't ask a base model "A, B, or C?" and trust the text it generates — it doesn't follow instructions. So benchmarks like **ARC, HellaSwag, MMLU** score base models a different way, with **no generation at all**:

> Form the full string `context + choice` for **each** candidate answer, and ask the model how **likely** it finds that continuation. Pick the choice with the highest log-likelihood. If that's the right answer, the model "knew" it.

The log-likelihood of a choice is the sum of the log-probabilities the model assigns to each of the choice's tokens, given everything before it:

```
context = "The sky is"      choice = " blue"  ->  tokens [blue]
LL = log P(blue | "The sky is")          # one term here; sum over all choice tokens in general
```

**Length normalization.** Longer choices have more (negative) log-prob terms, so raw LL unfairly favors short answers. We divide by the number of choice tokens — the standard "byte/token-length-normalized" accuracy.

| variable | meaning |
|---|---|
| `ctx_ids` | token ids of the context/prompt |
| `choice_ids` | token ids of one candidate completion |
| log-likelihood | `Σ log P(choice token | everything before it)` |

In [ ]:
# a tiny, easy benchmark (the mechanism; real benchmarks load the same shape from HF).
# Each: context, list of choices (with leading space), index of the correct one.
BENCH = [
    {"context": "The opposite of hot is", "choices": [" cold", " warm", " loud"], "answer": 0},
    {"context": "Water is made of hydrogen and", "choices": [" oxygen", " carbon", " iron"], "answer": 0},
    {"context": "The sun rises in the", "choices": [" east", " west", " south"], "answer": 0},
    {"context": "Two plus two equals", "choices": [" four", " seven", " ten"], "answer": 0},
    {"context": "A dog says", "choices": [" woof", " meow", " moo"], "answer": 0},
    {"context": "The first month of the year is", "choices": [" January", " June", " October"], "answer": 0},
    {"context": "Ice is frozen", "choices": [" water", " smoke", " sand"], "answer": 0},
    {"context": "The capital of France is", "choices": [" Paris", " Berlin", " Madrid"], "answer": 0},
]
# To use a REAL benchmark instead:
#   from datasets import load_dataset
#   d = load_dataset("ai2_arc", "ARC-Easy", split="validation")   # map fields into {context, choices, answer}
print(f"{len(BENCH)} questions, random-guess accuracy = {np.mean([1/len(b['choices']) for b in BENCH]):.2f}")

In [ ]:
@torch.no_grad()
def choice_loglikelihood(model, ctx_ids, choice_ids):
    """Sum of log P(token | preceding) over the CHOICE tokens. ctx_ids non-empty.

    Steps:
      1. ids = (ctx_ids + choice_ids), truncated to block_size, as a (1, L) tensor on device
      2. logits, _ = model(ids); logp = log_softmax(logits[0].float(), dim=-1)   # (L, V)
      3. the token at absolute position p is predicted by logp[p-1]. For choice token j
         (0-indexed within the choice), its position is len(ctx_ids)+j, predicted by logp[len(ctx)+j-1].
         Sum logp[len(ctx)+j-1, choice_ids[j]] over all j.
      4. return that scalar (a Python float)
    """
    # TODO:
    raise NotImplementedError


def evaluate_mc(model, bench):
    """Length-normalized log-likelihood accuracy over the benchmark (given)."""
    correct = 0
    for ex in bench:
        ctx = enc.encode_ordinary(ex["context"])
        scores = []
        for ch in ex["choices"]:
            cids = enc.encode_ordinary(ch)
            ll = choice_loglikelihood(model, ctx, cids)
            scores.append(ll / max(1, len(cids)))      # length-normalize
        correct += int(np.argmax(scores) == ex["answer"])
    return correct / len(bench)

In [ ]:
# check -- (a) the function matches a manual full-sequence computation; (b) report benchmark accuracy
# (a) mechanics check on a random model: our targeted sum must equal a manual log_softmax gather
torch.manual_seed(0); _rm = LlamaGPT(ckpt["cfg"]).to(device).eval()
_ctx, _ch = enc.encode_ordinary("the cat sat on the"), enc.encode_ordinary(" mat")
_ll = choice_loglikelihood(_rm, _ctx, _ch)
_ids = torch.tensor([_ctx + _ch], device=device)
_lg, _ = _rm(_ids); _lp = F.log_softmax(_lg[0].float(), -1)
_manual = sum(_lp[len(_ctx)+j-1, t].item() for j, t in enumerate(_ch))
assert abs(_ll - _manual) < 1e-3, "choice_loglikelihood must match the manual gather"
del _rm

# (b) the real eval -- honest: a 50M base may be near random on some of these
acc = evaluate_mc(model, BENCH)
print(f"benchmark accuracy: {acc:.2f}  (random guess ~ 0.33)")
for ex in BENCH:
    ctx = enc.encode_ordinary(ex["context"])
    sc = [choice_loglikelihood(model, ctx, enc.encode_ordinary(c)) / max(1, len(enc.encode_ordinary(c))) for c in ex["choices"]]
    pick = ex["choices"][int(np.argmax(sc))]
    print(f"  {ex['context']!r:42} -> {pick!r:10} {'OK' if np.argmax(sc)==ex['answer'] else 'x'}")
print("ok: multiple-choice-by-log-likelihood works (this is how ARC/HellaSwag score base models)")

## 3. Decoding — same model, very different text

Generation has two stages: the model gives a probability distribution over the next token, and a **decoding strategy** turns that distribution into a choice. You've been using fixed `temperature + top_k` since Ch.2. The decoder is a powerful, free dial — and the cure for the repetition loops you saw.

The standard toolbox, applied to the logits before sampling:

- **temperature** `T` — divide logits by `T`. `T<1` sharpens (more confident, repetitive), `T>1` flattens (more random). `T→0` = greedy.
- **top-k** — keep only the `k` highest-logit tokens. Crude (fixed count).
- **top-p (nucleus)** — keep the *smallest set of tokens whose probabilities sum to ≥ p*. Adaptive: few tokens when the model is confident, many when it isn't. The modern default.
- **min-p** — keep tokens with `prob ≥ min_p × (max prob)`. Even simpler and robust: a relative floor tied to the top token.
- **repetition penalty** — divide the logit of any *already-generated* token by `r > 1`, discouraging loops (the fix for "AC AC AC").

We implement these as a single **logit filter**; sampling is then just softmax + multinomial.

| knob | effect |
|---|---|
| `temperature` | sharpen / flatten the whole distribution |
| `top_k` | hard cap on the number of candidates |
| `top_p` | adaptive nucleus by cumulative probability |
| `min_p` | relative probability floor vs the top token |
| `repetition_penalty` | down-weight tokens already in the output |

In [ ]:
def filter_logits(logits, temperature=1.0, top_k=None, top_p=None, min_p=None,
                  repetition_penalty=1.0, prev_ids=None):
    """Apply the decoding filters to a 1-D logits tensor (vocab,). Returns filtered logits.

    Order: repetition penalty -> temperature -> top_k -> top_p -> min_p.
    Steps:
      1. repetition: for each token id t in set(prev_ids):
           logits[t] = logits[t]/rp if logits[t] > 0 else logits[t]*rp     (rp = repetition_penalty)
      2. temperature: logits = logits / temperature
      3. top_k: if set, keep the k largest; set the rest to -inf
           v = topk(logits, k).values; logits[logits < v[-1]] = -inf
      4. top_p (nucleus): if set (<1), sort desc, cumulative softmax probs;
           remove tokens whose cumulative prob exceeds top_p, but SHIFT the mask right by one
           (so the token that crosses p is kept) and always keep the top-1.
      5. min_p: if set, probs = softmax(logits); remove where probs < min_p * probs.max()
      6. return logits
    """
    # TODO:
    raise NotImplementedError


def sample_next(logits, **kw):
    probs = F.softmax(filter_logits(logits, **kw), dim=-1)
    return torch.multinomial(probs, 1)

In [ ]:
# check -- each filter does what it claims, on hand-built logits
import torch.nn.functional as _F
# repetition penalty lowers a positive logit of a previously-seen token
_lg = torch.tensor([2.0, 1.0, 0.5, -1.0])
_f = filter_logits(_lg.clone(), repetition_penalty=2.0, prev_ids=[0])
assert _f[0] < _lg[0] and torch.allclose(_f[1:], _lg[1:]), "rep penalty should only lower seen-token logits"

# top_p keeps the nucleus: probs ~ [0.64,0.23,0.09,...]; p=0.8 keeps first two, drops the rest
_lg2 = torch.tensor([3.0, 2.0, 1.0, 0.0, -1.0])
_f2 = filter_logits(_lg2.clone(), top_p=0.8)
_keep = torch.isfinite(_f2)
assert _keep[0] and _keep[1] and not _keep[2:].any(), f"top_p=0.8 should keep exactly the top-2 here, got {_keep}"

# min_p drops tokens below min_p * max_prob
_f3 = filter_logits(_lg2.clone(), min_p=0.3)
_p = _F.softmax(_lg2, -1)
assert torch.isfinite(_f3[0]) and not torch.isfinite(_f3[-1]), "min_p should keep the top and drop the tail"

# top_k keeps exactly k
_f4 = filter_logits(_lg2.clone(), top_k=2)
assert torch.isfinite(_f4).sum() == 2, "top_k=2 must keep exactly 2 tokens"
print("ok: temperature/top_k/top_p/min_p/repetition_penalty all behave")

In [ ]:
# see it in action: same prompt + model, different decoders (incl. the repetition fix)
@torch.no_grad()
def generate_with(model, prompt, n=50, **decode):
    model.eval()
    ids = enc.encode_ordinary(prompt)
    cur = torch.tensor([ids], device=device)
    for _ in range(n):
        logits, _ = model(cur[:, -model.block_size:])
        nxt = sample_next(logits[0, -1], prev_ids=cur[0].tolist(), **decode)
        cur = torch.cat([cur, nxt.view(1, 1)], dim=1)
        if nxt.item() == EOT: break
    return enc.decode(cur[0, len(ids):].tolist())

def distinct2(text):                      # crude repetition metric: fraction of unique adjacent word-pairs
    w = text.split()
    if len(w) < 2: return 1.0
    bg = list(zip(w, w[1:])); return len(set(bg)) / len(bg)

torch.manual_seed(0)
P = "The history of the Roman Empire"
configs = {
    "greedy (T=0.01)":        dict(temperature=0.01),
    "temp 1.0":               dict(temperature=1.0),
    "top_k=40":               dict(temperature=0.8, top_k=40),
    "top_p=0.9":              dict(temperature=0.8, top_p=0.9),
    "min_p=0.05":             dict(temperature=0.8, min_p=0.05),
    "top_p=0.9 + rep 1.3":    dict(temperature=0.8, top_p=0.9, repetition_penalty=1.3),
}
for name, dc in configs.items():
    txt = generate_with(model, P, n=40, **dc)
    print(f"[{name:22}] distinct-2={distinct2(txt):.2f}  {txt!r}")
    print()

## Done — you can now measure, not guess

| tool | what it answers | gotcha |
|---|---|---|
| **perplexity** | "how surprised is the model by real text?" | only fair on **held-out** data; use a reserved split |
| **MC log-likelihood** | "does the model *know* the answer?" (no generation) | length-normalize; a 50M base scores near random on hard sets — that's honest, not broken |
| **decoding** | "given the distribution, what text comes out?" | top-p / min-p + a small repetition penalty fixes loops without going incoherent |

A real eval setup (e.g. EleutherAI's **lm-evaluation-harness**) is exactly these primitives at scale: perplexity over corpora, log-likelihood over thousands of multiple-choice items, standardized decoding. You just built the core of it.

### The arc from here
Now that change is **measurable**:
- **Ch.8 — DPO:** preference-tune the chat model; measure the win with the MC/perplexity harness and an A/B on generations.
- **Ch.9 — Multilingual + tokenizer:** train a tokenizer, pretrain on language slices; measure per-language perplexity (and watch GPT-2 BPE's awful non-English "fertility").
- **Ch.10 — Quantization:** shrink the model to int8/int4; use perplexity to prove quality barely moves while memory drops.